# Viral Clipper on Colab (GPU runner)
Runtime > Change runtime type > **T4 GPU** first. Upload `viral-clipper.zip` to your Google Drive root (or clone your Git repo instead).

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/clipper_workspace'   # downloads, transcripts, outputs persist here

In [ ]:
# Option A: unzip from Drive
!rm -rf /content/viral-clipper && unzip -q -o /content/drive/MyDrive/viral-clipper.zip -d /content
%cd /content/viral-clipper
# Option B: git clone https://github.com/<you>/viral-clipper.git

In [ ]:
!apt-get -qq install -y fonts-liberation > /dev/null
!pip -q install -e ".[all]"
!clipper doctor

In [ ]:
# Optional: enables LLM ranking. Without it, heuristic scoring is used.
import os, getpass
os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (blank to skip): ')

In [ ]:
URL = 'https://www.youtube.com/watch?v=REPLACE_ME'
!clipper run "$URL" --work-dir "$WORK" --whisper-model large-v3 --clips 10
# YouTube may block Colab IPs ("confirm you're not a bot"). Fix: add --cookies /content/drive/MyDrive/cookies.txt,
# or download the video locally and pass its Drive path instead of a URL.

In [ ]:
import json, glob
from IPython.display import Video, display
meta = sorted(glob.glob(f'{WORK}/outputs/*/clips.json'), key=os.path.getmtime)[-1]
res = json.load(open(meta))
for c in res['clips']:
    print(f"#{c['rank']}  score {c['score']}  {c['start']:.0f}-{c['end']:.0f}s  {c['title']}")
    display(Video(c['video_path'], embed=False, width=270))